In [ ]:
# -*- coding: utf-8 -*-
# ==============================================================================
# TRAINING-FREE CERTIFIED BOUNDS FOR QUANTUM REGRESSION (PAPER-ALIGNED, COMPLETE)
# ==============================================================================
# - Computes true MSE_axis via full Pauli scan (d = 4^n) for n=4
# - Adaptive-only certified procedure (Hoeffding) + interpretability fields:
#     t_used_adaptive, stop_reason, p_L_final (and p_hat_final, s_successes)
# - Convergence credibility check for n=4:
#     t_epsilon (first random-scan hit within eps of optimum)
# - FIX: data_map_func is ParameterExpression-safe (symbolic-friendly)
# - Saves CSV + table PNGs
# ==============================================================================

import os
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# Qiskit (Hard requirement)
try:
    from qiskit.circuit.library import PauliFeatureMap
    from qiskit.quantum_info import SparsePauliOp, Statevector
except ImportError as e:
    raise ImportError(
        "\n\n[CRITICAL ERROR] Qiskit not found.\n"
        "Install via: pip install qiskit\n"
    ) from e


# ==============================================================================
# EXECUTION MODE
# ==============================================================================
# MODE = 1 -> FULL grid (3 datasets * 324 configs)
# MODE = 2 -> CUSTOM anchors only
MODE = 2


# ==============================================================================
# GLOBAL CONFIG
# ==============================================================================
SEED = 42
N_QUBITS = 5
D_FULL = 4 ** N_QUBITS  # 256

# data sizes
N_SYN = 200
N_REAL = 300

# -------------------- Threshold selection --------------------
# Paper: tau = Var(y) * (1 - R2_target)
TAU_MODE = "ratio"     # "r2" or "ratio"
R2_TARGET = 0.95    # used if TAU_MODE="r2"

# Practical alternative: tau = TAU_RATIO * Var(y)
TAU_RATIO = 0.95    # used if TAU_MODE="ratio"
# -------------------------------------------------------------

DELTA_TOTAL = 0.05
EPSILON_GAP = 1e-2  # for t_epsilon credibility

# adaptive sampling params
T0_PILOT = 50
BATCH_SIZE = 20
T_MAX = D_FULL

# futility tuning
EPS_MIN = 0.15
T_FUTILITY_CAP = 128

# trained validation
RIDGE_ALPHA = 1e-3

# outputs
OUTPUT_ROOT = os.path.join("results", f"MC_Repro_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)


# ==============================================================================
# DATASETS
# ==============================================================================
def target_correlated_gaussian(x: np.ndarray) -> float:
    n = len(x)
    cov = 0.3 * np.ones((n, n)) + 0.7 * np.eye(n)
    inv = np.linalg.inv(cov)
    return float(np.exp(-0.5 * x.T @ inv @ x))

def target_sparse_high_dim(x: np.ndarray, sparsity: float = 0.3) -> float:
    rng = np.random.default_rng(SEED)  # fixed subset for reproducibility
    n = len(x)
    idx = rng.choice(n, size=max(1, int(sparsity * n)), replace=False)
    mask = np.ones(n, dtype=bool)
    mask[idx] = False
    return float(np.sum(np.sin(x[idx])) + 0.1 * np.sum(x[mask]))

# ==============================================================================
# PREPROCESSORS
# ==============================================================================
def preproc_identity(X: np.ndarray) -> np.ndarray:
    return X

def preproc_tanh(X: np.ndarray) -> np.ndarray:
    return np.tanh(X)

def preproc_rbf_s1(X: np.ndarray) -> np.ndarray:
    return np.exp(-(X ** 2) / 2.0)

PREPROCESSORS = {
    "id": preproc_identity,
    "tanh": preproc_tanh,
    "rbf-s1": preproc_rbf_s1,
}


# ==============================================================================
# DATA-MAPPING RULES — SYMBOLIC-FRIENDLY (ParameterExpression-safe)
# ==============================================================================
def _sym_prod(v):
    p = 1
    for t in v:
        p = p * t
    return p

def dm_prod(v):
    return _sym_prod(v)

def dm_pi_prod(v):
    return np.pi * _sym_prod(v)

def dm_sum_plus_prod(v):
    return sum(v) + _sym_prod(v)  # Python sum only (no numpy), no float()

DATA_MAP_FUNCS = {
    "prod": dm_prod,
    "pi*prod": dm_pi_prod,
    "sum+prod": dm_sum_plus_prod,
}


# ==============================================================================
# FEATURE MAP CONFIG SPACE
# ==============================================================================
PAULI_SEQUENCES = {
    "Z+ZZ": ["Z", "ZZ"],
    "Y+YY": ["Y", "YY"],
    "X+XX": ["X", "XX"],
    "X,Y,Z": ["XZY", "YZX", "ZXY"],
    "all_pairs": ["YY", "ZZ", "XY", "XZ", "YZ"],
}
ENTANGLEMENT_OPTS = ["linear", "full"]
REPS_OPTS = [1, 2]


# ==============================================================================
# FULL PAULI BASIS (d=4^n)
# ==============================================================================
def full_pauli_labels(n_qubits: int) -> List[str]:
    from itertools import product
    alphabet = ["I", "X", "Y", "Z"]
    return ["".join(p) for p in product(alphabet, repeat=n_qubits)]


# ==============================================================================
# QUANTUM FEATURE MATRIX A (N x 4^n)
# ==============================================================================
def build_pauli_matrix_full(
    X: np.ndarray,
    pauli_seq_key: str,
    data_map_key: str,
    entanglement: str,
    reps: int,
    ops: List[SparsePauliOp],
) -> np.ndarray:
    n_samples, n = X.shape
    assert n == N_QUBITS, f"Expected n_features=n_qubits={N_QUBITS}, got {n}"

    fm = PauliFeatureMap(
        feature_dimension=n,
        paulis=PAULI_SEQUENCES[pauli_seq_key],
        entanglement=entanglement,
        reps=reps,
        data_map_func=DATA_MAP_FUNCS[data_map_key],
    )

    A = np.zeros((n_samples, len(ops)), dtype=float)

    for i, x in enumerate(X):
        bound = fm.assign_parameters(x, inplace=False)
        psi = Statevector.from_instruction(bound)
        for j, op in enumerate(ops):
            A[i, j] = float(np.real(psi.expectation_value(op)))

    return A


# ==============================================================================
# AXIS MSEs
# ==============================================================================
def calculate_axis_mses(A: np.ndarray, y: np.ndarray) -> Tuple[float, np.ndarray, float]:
    A = np.asarray(A, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    N, d = A.shape

    y_c = y - y.mean()
    var_y = float((y_c @ y_c) / N)
    if var_y < 1e-12:
        all_mses = np.zeros(d, dtype=float)
        return 0.0, all_mses, var_y

    A_c = A - A.mean(axis=0, keepdims=True)
    var_A = (A_c * A_c).sum(axis=0) / N
    cov = (A_c.T @ y_c) / N

    r2 = np.zeros_like(var_A)
    mask = var_A > 1e-12
    r2[mask] = (cov[mask] ** 2) / (var_A[mask] * var_y)

    all_mses = var_y * (1.0 - r2)
    mse_axis_true = float(np.min(all_mses))
    return mse_axis_true, all_mses, var_y


# ==============================================================================
# Convergence credibility: t_epsilon (n=4)
# ==============================================================================
def calculate_t_epsilon(all_mses: np.ndarray, mse_true: float, epsilon: float = EPSILON_GAP, seed: int = SEED) -> int:
    rng = np.random.default_rng(seed)
    d = len(all_mses)
    thr = mse_true + epsilon

    perm = rng.permutation(d)
    running_min = np.inf
    for t, idx in enumerate(perm, start=1):
        running_min = min(running_min, float(all_mses[idx]))
        if running_min <= thr:
            return t
    return d


# ==============================================================================
# Adaptive certified MC (Hoeffding)
# ==============================================================================
def adaptive_mc_certified(
    all_mses: np.ndarray,
    tau: float,
    delta_total: float = DELTA_TOTAL,
    t0: int = T0_PILOT,
    batch: int = BATCH_SIZE,
    t_max: int = T_MAX,
    eps_min: float = EPS_MIN,
    t_futility_cap: int = T_FUTILITY_CAP,
    seed: int = SEED,
) -> Dict[str, Any]:

    rng = np.random.default_rng(seed)
    d = len(all_mses)

    alpha = delta_total / 2.0
    delta = delta_total / 2.0

    t0 = min(max(1, int(t0)), d)
    batch = max(1, int(batch))
    t_max = min(max(1, int(t_max)), d)
    t_futility_cap = min(max(t0, int(t_futility_cap)), t_max)

    # initial sample
    T = list(rng.choice(d, size=t0, replace=False))
    used = set(T)
    t = len(T)
    s = int(np.sum(all_mses[T] <= tau))     # successes
    mse_hat = float(np.min(all_mses[T]))    # MC bound

    remaining = np.array([i for i in range(d) if i not in used], dtype=int)
    rng.shuffle(remaining)
    rem_ptr = 0

    last_p_L = 0.0
    last_t_req = np.inf
    stop_reason = "budget"

    while True:
        p_hat = s / t
        eps = float(np.sqrt(np.log(1.0 / alpha) / (2.0 * t)))
        p_L = max(0.0, p_hat - eps)

        if p_L > 0.0:
            denom = np.log(1.0 / (1.0 - p_L))
            t_req = float(np.log(1.0 / delta) / denom) if denom > 1e-12 else np.inf
        else:
            t_req = np.inf

        last_p_L = float(p_L)
        last_t_req = float(t_req)

        # (1) certified
        if t >= t_req:
            stop_reason = "certified"
            break

        # (2) futility (no successes so far)
        if p_hat == 0.0:
            if eps < eps_min:
                stop_reason = "futility_eps"
                break
            if t >= t_futility_cap:
                stop_reason = "futility_cap"
                break

        # (3) budget / exhaustion
        if t >= t_max or rem_ptr >= len(remaining):
            stop_reason = "budget"
            break

        # next batch
        m = min(batch, t_max - t, len(remaining) - rem_ptr)
        new_axes = remaining[rem_ptr: rem_ptr + m].tolist()
        rem_ptr += m

        t += m
        s += int(np.sum(all_mses[new_axes] <= tau))
        mse_hat = min(mse_hat, float(np.min(all_mses[new_axes])))

    return {
        "t_used": int(t),
        "t_req_pred": (int(np.ceil(last_t_req)) if np.isfinite(last_t_req) else int(d)),
        "p_L_final": float(last_p_L),
        "p_hat_final": float(s / t),
        "s_successes": int(s),
        "mse_hat_adaptive": float(mse_hat),
        "stop_reason": stop_reason,
    }


# ==============================================================================
# TABLE PLOTTER (PNG)
# ==============================================================================
def save_table_png(
    df: pd.DataFrame,
    out_path: str,
    title: str,
    cols: List[str],
    max_rows: int = 30,
    float_fmt: str = "{:.4f}",
) -> None:
    if df.empty:
        return

    dff = df.copy()
    if "mse_axis_true" in dff.columns:
        dff = dff.sort_values("mse_axis_true", ascending=True)

    if len(dff) > max_rows:
        dff = dff.head(max_rows)

    dff = dff[cols].copy()

    for c in dff.columns:
        if pd.api.types.is_float_dtype(dff[c]):
            dff[c] = dff[c].map(lambda x: float_fmt.format(x))

    fig_h = 1.0 + 0.28 * (len(dff) + 1)
    fig_w = max(12.0, 0.9 * len(cols) + 6.0)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    ax.set_title(title, fontsize=14, pad=12)

    table = ax.table(
        cellText=dff.values,
        colLabels=dff.columns.tolist(),
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.3)

    plt.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def load_riemann_features():
    features = pd.read_csv("../results/data_analysis/selected_features.csv")["feature"].tolist()
    dataset = pd.read_csv("../dataset/riemann_features.csv")

    features = features[:N_QUBITS]  # limit to N_QUBITS features for consistency

    X = dataset.drop(columns=["distance"])[features].to_numpy()
    y = dataset["distance"].to_numpy()
    X = X[:2000]
    y = y[:2000]

    return X, y


# ==============================================================================
# CONFIG BUILDERS
# ==============================================================================
def build_full_grid() -> List[Dict[str, Any]]:
    configs = []
    for pp in PREPROCESSORS.keys():
        for pm in PAULI_SEQUENCES.keys():
            for dm in DATA_MAP_FUNCS.keys():
                for ent in ENTANGLEMENT_OPTS:
                    for r in REPS_OPTS:
                        configs.append({
                            "preprocessor": pp,
                            "data_map": dm,
                            "pauli_seq": pm,
                            "entanglement": ent,
                            "reps": r,
                        })
    return configs


def build_anchor_configs() -> List[Dict[str, Any]]:
    return [
        {"dataset": "real_california", "preprocessor": "rbf-s1", "data_map": "prod",
         "pauli_seq": "Z+ZZ", "entanglement": "linear", "reps": 2},
        {"dataset": "syn_corr_gauss", "preprocessor": "rbf-s1", "data_map": "prod",
         "pauli_seq": "Y+YY", "entanglement": "linear", "reps": 1},
        {"dataset": "syn_sparse", "preprocessor": "tanh", "data_map": "prod",
         "pauli_seq": "Z+ZZ", "entanglement": "full", "reps": 1},
    ]


# ==============================================================================
# THRESHOLD tau
# ==============================================================================
def compute_tau(var_y: float) -> Tuple[float, float]:
    if TAU_MODE == "r2":
        tau = float(var_y * (1.0 - R2_TARGET))
        return tau, float(R2_TARGET)
    elif TAU_MODE == "ratio":
        tau = float(TAU_RATIO * var_y)
        r2_eff = float(1.0 - (tau / var_y)) if var_y > 1e-12 else 0.0
        return tau, r2_eff
    else:
        raise ValueError("TAU_MODE must be 'r2' or 'ratio'.")


# ==============================================================================
# EVALUATE ONE CONFIG
# ==============================================================================
def evaluate_config(cfg: Dict[str, Any], ops: List[SparsePauliOp]) -> Dict[str, Any]:
    preproc = cfg["preprocessor"]
    pauli_seq = cfg["pauli_seq"]
    data_map = cfg["data_map"]
    ent = cfg["entanglement"]
    reps = int(cfg["reps"])

    X, y = load_riemann_features()

    # preprocess X
    Xp = PREPROCESSORS[preproc](X)

    # quantum features (full basis)
    A = build_pauli_matrix_full(
        X=Xp,
        pauli_seq_key=pauli_seq,
        data_map_key=data_map,
        entanglement=ent,
        reps=reps,
        ops=ops,
    )

    # true axis bound
    mse_true, all_mses, var_y = calculate_axis_mses(A, y)

    # tau
    tau, r2_target_eff = compute_tau(var_y)

    # feasibility diagnostics
    r2_axis_max = float(1.0 - (mse_true / var_y)) if var_y > 1e-12 else 0.0
    target_unreachable = int(r2_target_eff > r2_axis_max + 1e-12)
    tau_minus_mse_axis = float(tau - mse_true)

    # credibility check
    t_eps = calculate_t_epsilon(all_mses, mse_true, epsilon=EPSILON_GAP, seed=SEED)

    # adaptive MC (main story)
    adapt = adaptive_mc_certified(
        all_mses=all_mses,
        tau=tau,
        delta_total=DELTA_TOTAL,
        t0=T0_PILOT,
        batch=BATCH_SIZE,
        t_max=T_MAX,
        eps_min=EPS_MIN,
        t_futility_cap=T_FUTILITY_CAP,
        seed=SEED,
    )

    mse_mc_adapt = float(adapt["mse_hat_adaptive"])
    gap_adapt = float(mse_mc_adapt - mse_true)

    # trained validation (optional, but useful in results section)
    ridge = Ridge(alpha=RIDGE_ALPHA).fit(A, y)
    ridge_mse = float(mean_squared_error(y, ridge.predict(A)))

    svr = SVR().fit(X, y)
    svr_mse = float(mean_squared_error(y, svr.predict(X)))

    return {
        "n_qubits": N_QUBITS,
        "d_full": int(len(all_mses)),

        "preprocessor": preproc,
        "data_map": data_map,
        "pauli_seq": pauli_seq,
        "entanglement": ent,
        "reps": reps,

        "var_y": float(var_y),
        "tau_mode": TAU_MODE,
        "r2_target_effective": float(r2_target_eff),
        "tau": float(tau),

        "r2_axis_max": float(r2_axis_max),
        "target_unreachable": int(target_unreachable),
        "tau_minus_mse_axis": float(tau_minus_mse_axis),

        "mse_axis_true": float(mse_true),

        "mse_mc_adaptive": float(mse_mc_adapt),
        "gap_mc_adaptive": float(gap_adapt),
        "t_used_adaptive": int(adapt["t_used"]),
        "t_req_pred": int(adapt["t_req_pred"]),
        "p_L_final": float(adapt["p_L_final"]),
        "p_hat_final": float(adapt["p_hat_final"]),
        "s_successes": int(adapt["s_successes"]),
        "stop_reason": str(adapt["stop_reason"]),

        "t_epsilon": int(t_eps),

        "ridge_train_mse": float(ridge_mse),
        "svr_classic_mse": float(svr_mse),
    }


# ==============================================================================
# RUNNER
# ==============================================================================
def run_experiments() -> None:
    configs = build_full_grid() if MODE == 1 else build_anchor_configs()
    print(f"Running MODE={MODE} with {len(configs)} configs...")
    print(f"TAU_MODE={TAU_MODE} | R2_TARGET={R2_TARGET} | TAU_RATIO={TAU_RATIO}")

    # precompute full Pauli basis once
    labels = full_pauli_labels(N_QUBITS)
    ops = [SparsePauliOp(lab[::-1]) for lab in labels]  # endian flip: OK

    results = []
    errors = []

    for i, cfg in enumerate(configs, start=1):
        tag = f"[{i}/{len(configs)}] | {cfg['preprocessor']} | {cfg['data_map']} | {cfg['pauli_seq']} | {cfg['entanglement']} | r={cfg['reps']}"
        try:
            print(tag)
            res = evaluate_config(cfg, ops)
            results.append(res)
        except Exception as e:
            msg = f"ERROR on {tag}: {e}"
            print(msg)
            errors.append({"config": tag, "error": str(e)})

    df = pd.DataFrame(results)
    err_df = pd.DataFrame(errors)

    out_csv = os.path.join(OUTPUT_ROOT, "results_final.csv")
    df.to_csv(out_csv, index=False)
    print(f"\nSaved results to: {out_csv}")

    if not err_df.empty:
        out_err = os.path.join(OUTPUT_ROOT, "errors.csv")
        err_df.to_csv(out_err, index=False)
        print(f"Saved errors to: {out_err}")

    # ---------- Table PNG (Top rows) ----------
    if not df.empty:
        cols_main = [
            "dataset", "preprocessor", "data_map", "pauli_seq", "entanglement", "reps",
            "mse_axis_true",
            "mse_mc_adaptive", "gap_mc_adaptive",
            "t_used_adaptive", "stop_reason",
            "p_L_final", "p_hat_final", "s_successes",
            "tau_mode", "tau", "r2_target_effective",
            "r2_axis_max", "target_unreachable", "tau_minus_mse_axis",
            "t_epsilon",
            "ridge_train_mse",
        ]
        table_png = os.path.join(OUTPUT_ROOT, "results_table_top.png")
        save_table_png(
            df=df,
            out_path=table_png,
            title="Resultados (Top por MSE_axis_true) — Adaptive MC + t_epsilon",
            cols=cols_main,
            max_rows=30,
            float_fmt="{:.4f}",
        )
        print(f"Saved table image to: {table_png}")

    # ---------- Aggregate Table (MODE 1) ----------
    if MODE == 1 and not df.empty:
        d_over_4 = int(D_FULL // 4)

        agg = df.groupby("dataset").agg(
            mean_t_epsilon=("t_epsilon", "mean"),
            median_t_epsilon=("t_epsilon", "median"),
            min_t_epsilon=("t_epsilon", "min"),
            prob_t_eps_le_d4=("t_epsilon", lambda x: float(np.mean(x <= d_over_4))),

            mean_t_used_adapt=("t_used_adaptive", "mean"),
            median_t_used_adapt=("t_used_adaptive", "median"),
            frac_certified=("stop_reason", lambda s: float(np.mean(np.array(s) == "certified"))),
            frac_futility=("stop_reason", lambda s: float(np.mean(np.isin(np.array(s), ["futility_eps", "futility_cap"])))),
            frac_budget=("stop_reason", lambda s: float(np.mean(np.array(s) == "budget"))),

            mean_gap_mc_adapt=("gap_mc_adaptive", "mean"),
            median_gap_mc_adapt=("gap_mc_adaptive", "median"),
            mean_pL=("p_L_final", "mean"),
            mean_phat=("p_hat_final", "mean"),

            frac_unreachable=("target_unreachable", "mean"),
        ).reset_index()

        agg_csv = os.path.join(OUTPUT_ROOT, "table1_aggregate.csv")
        agg.to_csv(agg_csv, index=False)
        print(f"Saved aggregate csv to: {agg_csv}")

        agg_png = os.path.join(OUTPUT_ROOT, "table1_aggregate.png")
        save_table_png(
            df=agg,
            out_path=agg_png,
            title=f"Tabela 1 (Aggregate) — Adaptive MC + Convergência (t_epsilon, d/4={d_over_4})",
            cols=agg.columns.tolist(),
            max_rows=20,
            float_fmt="{:.6f}",
        )
        print(f"Saved aggregate table image to: {agg_png}")

    # ---------- Quick console view (Anchors) ----------
    if MODE == 2 and not df.empty:
        cols_preview = [
            "dataset",
            "mse_axis_true",
            "mse_mc_adaptive", "gap_mc_adaptive",
            "t_used_adaptive", "stop_reason", "p_L_final",
            "t_epsilon",
            "r2_axis_max", "target_unreachable",
            "ridge_train_mse",
        ]
        print("\nANCHOR SUMMARY:")
        print(df[cols_preview].to_string(index=False))


if __name__ == "__main__":
    run_experiments()


Running MODE=2 with 3 configs...
TAU_MODE=ratio | R2_TARGET=0.95 | TAU_RATIO=0.95
[1/3] | rbf-s1 | prod | Z+ZZ | linear | r=2


/tmp/ipykernel_561447/3865786532.py:188: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation.pauli_feature_map.PauliFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the pauli_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  fm = PauliFeatureMap(


KeyboardInterrupt: 